# Stage 4 — Sparse Jump Model (SJM) regime signal

Reproduction of Stage 4 of **Shu & Mulvey (2024)**, *Dynamic Factor Allocation Leveraging Regime-Switching Signals*.

Fits the author's **Sparse Jump Model** (`jumpmodels` library) per factor and produces a daily bull/bear regime signal, then evaluates signal quality with the paper's single-factor long-short test (Exhibit 6).

**Steps** (run top to bottom; each step prints a sanity summary and there is a `PAUSE` marker where the spec asks you to confirm before continuing):
1. Load & confirm inputs.
2. Train-only preprocessing utility (clip + standardize).
3. Single prototype fit on momentum (fixed hyper-params), in-sample regimes.
4. Online walk-forward with monthly refit → `signals.parquet` (all 6 factors).
5. Long-short signal-quality evaluation → `longshort_eval.parquet`.

**Strictly no look-ahead:** scaler + SJM are fit on training data only; the live signal is shifted forward before it is used.

Black-Litterman (Stage 6) is intentionally **not** built here.

In [ ]:
# ----------------------------------------------------------------------
# 0. Imports & configuration
# ----------------------------------------------------------------------
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from importlib.metadata import version

from jumpmodels.preprocess import StandardScalerPD, DataClipperStd
from jumpmodels.sparse_jump import SparseJumpModel

warnings.filterwarnings("ignore")
print("jumpmodels version:", version("jumpmodels"))

# --- The 6 factors ---
FACTORS = ["value", "size", "momentum", "quality", "lowvol", "growth"]

# --- Stage-3 outputs live in this folder (root), not a features/ subdir ---
DATA_DIR = "."

# --- SJM hyper-parameters (fixed for Stage 4; tuning is a later step) ---
N_COMPONENTS = 2
JUMP_PENALTY = 50.0
MAX_FEATS    = 10.0
RANDOM_STATE = 0

# --- Preprocessing ---
CLIP_MUL = 3.0          # DataClipperStd: clip raw features at +/- 3 std (train std)

# --- Walk-forward (Step 4) ---
TEST_START          = pd.Timestamp("2007-01-01")  # 1999-2006 = burn-in
MAX_LOOKBACK_YEARS  = 12                          # cap the expanding window at 12y
MIN_TRAIN_YEARS     = 8                           # nominal minimum train length
REFIT_FREQ          = "M"                          # refit monthly ('Q' = quarterly, faster)
SIGNAL_DELAY        = 2                            # regime inferred end-of-T used from T+2

# --- Evaluation (Step 5) ---
ANNUAL        = 252
RET_CAP       = 0.05      # +/-5%/yr expected active return -> +/-100% position
COST_PER_SIDE = 0.0005   # 5 bps per side on position changes

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

## Step 1 — Load and confirm
Load all 6 feature tables + `active_returns.parquet`; print shapes, date ranges, confirm 0 NaNs.

`active_returns.parquet` keeps its raw scattered NaNs (momentum ~22, growth ~250). For every use here it is re-indexed to a factor's feature index and missing days are filled with `0.0` (no active move), matching the Stage-3 treatment.

In [ ]:
# Feature tables: one DataFrame per factor (DatetimeIndex 'date', 17 cols, 0 NaN).
features = {f: pd.read_parquet(f"{DATA_DIR}/{f}.parquet") for f in FACTORS}

# Raw active returns (factor - market).
active_raw = pd.read_parquet(f"{DATA_DIR}/active_returns.parquet")

# Per-factor active returns aligned to that factor's feature index, gaps -> 0.
active = {f: active_raw[f].reindex(features[f].index).fillna(0.0) for f in FACTORS}

print("==== FEATURE TABLES ====")
for f in FACTORS:
    df = features[f]
    print(f"{f:9s} shape={df.shape}  {df.index.min().date()} -> {df.index.max().date()}  NaNs={int(df.isna().sum().sum())}")
print("\nfeature columns:", list(features['value'].columns))
print("\n==== ACTIVE RETURNS (raw) ====")
print("shape", active_raw.shape, active_raw.index.min().date(), '->', active_raw.index.max().date())
print("raw NaN per col:\n", active_raw.isna().sum().to_dict())
print("aligned+filled NaN total:", int(sum(active[f].isna().sum() for f in FACTORS)))

> **PAUSE (Step 1).** Confirm 6 tables load, 0 NaNs, dates ~1999-03-31 → 2026-06-15, library installed.

## Step 2 — Preprocessing utility (train-only, no look-ahead)
Clip raw features at `±CLIP_MUL` std then standardize. **Both the clipper and the scaler are fit on the TRAIN slice only** and merely applied to test. Returns scaled train/test plus the fitted transformers.

In [ ]:
def scale_train_test(X_train, X_test=None, clip_mul=CLIP_MUL):
    """Fit DataClipperStd + StandardScalerPD on X_train only; apply to both.
    Returns (X_train_scaled, X_test_scaled_or_None, clipper, scaler)."""
    clipper = DataClipperStd(mul=clip_mul)
    scaler  = StandardScalerPD()
    X_tr = scaler.fit_transform(clipper.fit_transform(X_train))   # fit on TRAIN
    X_te = None
    if X_test is not None and len(X_test):
        X_te = scaler.transform(clipper.transform(X_test))        # apply only
    return X_tr, X_te, clipper, scaler


# --- sanity test on momentum over a sample train window ---
_Xtr = features["momentum"].loc[:"2012-12-31"]
_Xte = features["momentum"].loc["2013-01-01":"2013-12-31"]
_Xtr_s, _Xte_s, _, _ = scale_train_test(_Xtr, _Xte)
print("train scaled shape:", _Xtr_s.shape, " test scaled shape:", _Xte_s.shape)
print("TRAIN  max|mean| = %.4f   mean(std) = %.4f   (target: ~0 and ~1)" % (
    _Xtr_s.mean().abs().max(), _Xtr_s.std().mean()))
print("TEST   max|mean| = %.4f   mean(std) = %.4f   (should drift off 0/1: out-of-sample)" % (
    _Xte_s.mean().abs().max(), _Xte_s.std().mean()))
_Xtr_s.describe().T[["mean", "std", "min", "max"]].round(3)

> **PAUSE (Step 2).** Confirm TRAIN columns are mean~0 / std~1.

## Step 3 — Single prototype fit (momentum, fixed hyper-params)
Train window 1999-01 → 2012-12. Fit `SparseJumpModel(n_components=2, jump_penalty=50, max_feats=10, random_state=0)` on scaled training features. Then label bull vs bear, report state stats, and plot in-sample regimes.

In [ ]:
def bull_bear_map(labels, ret):
    """Map raw SJM states -> {0=bear, 1=bull}. Bull = state with the higher
    CUMULATIVE active return on its assigned days. Always returns keys for both
    states 0 and 1 (robust to a degenerate single-state window)."""
    cum = {s: float(ret[labels == s].sum()) for s in (0, 1)}
    bull = max(cum, key=cum.get)
    return {s: (1 if s == bull else 0) for s in (0, 1)}


def regime_means_annual(mapped_labels, ret):
    """Annualized mean active return for each regime (0/1) on training days."""
    out = {}
    for g in (0, 1):
        r = ret[mapped_labels == g]
        out[g] = float(r.mean() * ANNUAL) if len(r) else 0.0
    return out


def n_switches(sig):
    """Number of regime switches in a 0/1 signal series."""
    s = pd.Series(sig).dropna()
    return int((s.diff().fillna(0) != 0).sum())


def shade_regimes(ax, sig):
    """Shade contiguous bull (green) / bear (red) runs on a time axis."""
    s = pd.Series(sig).dropna()
    grp = (s != s.shift()).cumsum()
    for _, run in s.groupby(grp):
        ax.axvspan(run.index[0], run.index[-1],
                   color=("green" if run.iloc[0] == 1 else "red"), alpha=0.15)

In [ ]:
# --- 3. fit momentum on 1999-01 .. 2012-12 ---
Xtr   = features["momentum"].loc[:"2012-12-31"]
rtr   = active["momentum"].loc[:"2012-12-31"]
Xtr_s, _, _, _ = scale_train_test(Xtr)

proto = SparseJumpModel(n_components=N_COMPONENTS, jump_penalty=JUMP_PENALTY,
                        max_feats=MAX_FEATS, random_state=RANDOM_STATE)
proto.fit(Xtr_s, ret_ser=rtr, sort_by="cumret")

# 3a. in-sample labels
raw_lab = pd.Series(np.asarray(proto.predict(Xtr_s)), index=Xtr_s.index)

# 3b. bull/bear mapping (higher cumulative active return = bull)
mapping = bull_bear_map(raw_lab, rtr)
lab     = raw_lab.map(mapping)            # 0=bear, 1=bull
print("raw-state -> {0:bear,1:bull} mapping:", mapping)

# 3c. state stats
rmeans = regime_means_annual(lab, rtr)
print("\nbull days:", int((lab == 1).sum()), "  bear days:", int((lab == 0).sum()))
print("avg annualized active return  | BULL: %+.2f%%   BEAR: %+.2f%%" % (
    rmeans[1] * 100, rmeans[0] * 100))
print("regime switches over train window:", n_switches(lab))
print("feature weights (nonzero):", int((proto.feat_weights > 1e-8).sum()), "of", len(proto.feat_weights))

In [ ]:
# 3d. plot in-sample regimes over cumulative momentum active return
cum = (1 + rtr).cumprod()
fig, ax = plt.subplots(figsize=(13, 6))
ax.plot(cum.index, cum.values, color="black", linewidth=1.1, label="cum. active return")
shade_regimes(ax, lab)
ax.set_title("Momentum — in-sample SJM regimes (green=bull, red=bear)  [1999–2012]")
ax.set_xlabel("date"); ax.set_ylabel("cumulative active growth of $1")
ax.legend(loc="upper left"); ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig("momentum_insample_regimes.png", dpi=150)
print("Saved momentum_insample_regimes.png")
plt.show()

> **PAUSE (Step 3).** Confirm regimes look persistent/sensible, bull clearly out-earns bear, and the shape resembles paper Exhibit 4.

## Step 4 — Online walk-forward with monthly refit (all 6 factors)

- **Expanding** training window from data start, **capped at 12 years** lookback (nominal min ~8y).
- **Refit the SJM monthly.** Within each month, `predict_online` infers the daily state on new data **without** refitting.
- **Bull/bear mapping** is recomputed from each refit's training data.
- **Look-ahead control:** the regime inferred at end of day T is applied from day **T+2** (`SIGNAL_DELAY=2`): the raw daily signal is shifted forward 2 trading days before use. *(The spec's general rule mentions "+1 day"; Step 4 specifies T+2, which is what is implemented. Change `SIGNAL_DELAY` to 1 if you prefer the looser rule.)*

> ⚠️ **Runtime:** ~240 monthly refits × 6 factors ≈ 1,400 SJM fits. Expect this cell to run for a number of minutes. For a quick trial set `REFIT_FREQ='Q'` in the config cell.

In [ ]:
def walk_forward(factor, verbose=True):
    """Monthly-refit online walk-forward for one factor.
    Returns RAW (unshifted) daily signal and expected-annual-active-return series
    over the test period. The +SIGNAL_DELAY shift is applied later, once."""
    X   = features[factor]
    ret = active[factor]
    idx = X.index
    test_idx = idx[idx >= TEST_START]

    raw_sig = pd.Series(index=test_idx, dtype=float)   # 0/1 inferred regime
    raw_exp = pd.Series(index=test_idx, dtype=float)   # ann. expected active ret

    periods = test_idx.to_period(REFIT_FREQ)
    for p in periods.unique():
        block = test_idx[periods == p]
        b0 = block[0]
        # training window: all data strictly before this block, capped at 12y
        lb_start  = b0 - pd.DateOffset(years=MAX_LOOKBACK_YEARS)
        train_idx = idx[(idx < b0) & (idx >= lb_start)]
        Xtr, rtr  = X.loc[train_idx], ret.loc[train_idx]
        Xte       = X.loc[block]

        # scale (fit on train only), fit SJM
        Xtr_s, Xte_s, _, _ = scale_train_test(Xtr, Xte)
        model = SparseJumpModel(n_components=N_COMPONENTS, jump_penalty=JUMP_PENALTY,
                                max_feats=MAX_FEATS, random_state=RANDOM_STATE)
        model.fit(Xtr_s, ret_ser=rtr, sort_by="cumret")

        # bull/bear mapping + per-regime annual mean from THIS train window
        tr_lab  = pd.Series(np.asarray(model.predict(Xtr_s)), index=Xtr_s.index)
        mp      = bull_bear_map(tr_lab, rtr)
        rmeans  = regime_means_annual(tr_lab.map(mp), rtr)

        # online inference: feed train+block, keep the block labels (causal/filtered)
        Xfull_s = pd.concat([Xtr_s, Xte_s])
        on      = pd.Series(np.asarray(model.predict_online(Xfull_s)), index=Xfull_s.index)
        on_blk  = on.loc[block].map(mp)
        raw_sig.loc[block] = on_blk.values
        raw_exp.loc[block] = on_blk.map(rmeans).values

    if verbose:
        print(f"  {factor:9s} done: {len(periods.unique())} refits, "
              f"{int(raw_sig.notna().sum())} signal days")
    return raw_sig, raw_exp

In [ ]:
# --- run walk-forward for all 6 factors (this is the slow cell) ---
raw_signals, raw_exp = {}, {}
print("Running monthly-refit walk-forward (REFIT_FREQ=%s, delay=%d) ..." % (REFIT_FREQ, SIGNAL_DELAY))
for f in FACTORS:
    raw_signals[f], raw_exp[f] = walk_forward(f)

raw_sig_df = pd.DataFrame(raw_signals)
raw_exp_df = pd.DataFrame(raw_exp)

# Apply the look-ahead-safe forward shift ONCE, then drop warmup rows.
signals    = raw_sig_df.shift(SIGNAL_DELAY).dropna().astype(int)
exp_active = raw_exp_df.shift(SIGNAL_DELAY).reindex(signals.index)

signals.to_parquet("signals.parquet")
exp_active.to_parquet("exp_active.parquet")
print("\nSaved signals.parquet", signals.shape, " and exp_active.parquet", exp_active.shape)
print("signal date range:", signals.index.min().date(), "->", signals.index.max().date())

In [ ]:
# --- Step 4 sanity summary: switches/yr and % days bull, per factor ---
n_years = (signals.index[-1] - signals.index[0]).days / 365.25
summary4 = pd.DataFrame({
    "switches_per_yr": {f: round(n_switches(signals[f]) / n_years, 2) for f in FACTORS},
    "pct_days_bull":   {f: round(signals[f].mean() * 100, 1) for f in FACTORS},
})
print("Test span: %.1f years (%s -> %s)" % (n_years, signals.index.min().date(), signals.index.max().date()))
print(summary4)

> **PAUSE (Step 4).** Confirm switch frequency is modest (persistent regimes, not whipsawing) and bull fraction is reasonable per factor.

## Step 5 — Signal-quality evaluation (single-factor long-short, Exhibit 6)

Per factor, a relative long-factor / short-market strategy:
- **expected active return** = train-window average active return under the currently inferred regime (annualized) — this is `exp_active` from Step 4.
- **position** `weight = clip(expected_active_return / 0.05, -1, +1)` (±5%/yr → ±100%).
- **daily strategy return** = `weight × active_return(factor)` minus **5 bps per side** on position changes.

Reports annualized Sharpe per factor (paper: ~0.16–0.39, all positive) and the 6×6 strategy-return correlation matrix (paper: mostly < 0.5).

In [ ]:
weights   = (exp_active / RET_CAP).clip(-1, 1)          # position per factor per day
strat_ret = pd.DataFrame(index=signals.index)
sharpe    = {}

for f in FACTORS:
    w = weights[f]
    r = active[f].reindex(w.index).fillna(0.0)          # realized active return
    turnover = w.diff().abs().fillna(w.abs())           # day-1 entry counts as turnover
    net = w * r - COST_PER_SIDE * turnover              # net daily strategy return
    strat_ret[f] = net
    sharpe[f] = net.mean() / net.std() * np.sqrt(ANNUAL)

sharpe_tbl = pd.Series(sharpe, name="ann_Sharpe").round(3)
corr = strat_ret.corr().round(2)

strat_ret.to_parquet("longshort_eval.parquet")
print("Saved longshort_eval.parquet", strat_ret.shape)
print("\n=== Annualized Sharpe per factor (paper ~0.16-0.39, all > 0) ===")
print(sharpe_tbl)
print("mean Sharpe:", round(sharpe_tbl.mean(), 3))
print("\n=== Strategy-return correlation matrix (paper: mostly < 0.5) ===")
print(corr)

---
### Stage 4 complete
Artifacts written to this folder:
- `momentum_insample_regimes.png` — Step 3 prototype regimes.
- `signals.parquet` — daily bull(1)/bear(0) signal, 6 factors, over the test period (look-ahead-safe).
- `exp_active.parquet` — per-day annualized expected active return by inferred regime (intermediate).
- `longshort_eval.parquet` — daily long-short strategy returns used for the Sharpe / correlation check.

**STOP here.** Black-Litterman (Stage 6) is not started.